# Défi quotidien : Optimiser un LLM avec LoRA (PEFT)

On va appliquer **LoRA** (Low-Rank Adaptation) à un modèle de langage pré-entraîné `bigscience/bloomz-560m`, en utilisant la librairie Hugging Face **PEFT**, sur un échantillon du dataset `Abirate/english_quotes`.

Étapes :
1. Installer les librairies nécessaires.
2. Charger le modèle pré-entraîné et son tokenizer.
3. Charger et prétraiter le dataset.
4. Configurer LoRA (`LoraConfig`).
5. Appliquer LoRA au modèle (`get_peft_model`).
6. Configurer l'entraînement (`TrainingArguments` + `Trainer`).
7. Entraîner, sauvegarder, puis recharger le modèle LoRA.
8. Générer du texte avec le modèle fine-tuné.


## Étape 1 : Installation des librairies

In [ ]:
%pip install --quiet peft==0.4.0 transformers datasets accelerate


In [ ]:
import os

os.makedirs("../cache", exist_ok=True)


## Étape 2 : Charger le modèle pré-entraîné et le tokenizer

On utilise `bigscience/bloomz-560m`, un modèle causal multilingue relativement léger, adapté pour un fine-tuning rapide sur CPU.

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"

tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

print(foundation_model)


## Étape 3 : Charger et prétraiter le dataset

On charge `Abirate/english_quotes`, on en garde 10% (split d'entraînement), puis on tokenize le champ `quote`.

In [ ]:
dataset = load_dataset("Abirate/english_quotes")

# Echantillon de 10% du split "train"
data = dataset["train"].shuffle(seed=42).select(range(int(0.1 * len(dataset["train"]))))

# Tokenization
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

train_sample = data.select(range(5))
display(train_sample)


## Étape 4 : Configurer LoRA avec `LoraConfig`

- `r` : rang des matrices de faible rang (plus petit = moins de paramètres entraînables).
- `lora_alpha` : facteur d'échelle appliqué à la mise à jour LoRA.
- `target_modules` : les couches du modèle ciblées par LoRA. Pour BLOOM, la couche d'attention combinée s'appelle `query_key_value`.
- `lora_dropout` : dropout appliqué aux activations LoRA pour la régularisation.
- `bias` : on ne rend pas les biais entraînables ("none").
- `task_type` : `CAUSAL_LM` car on fait de la génération de texte autoregressive.

In [ ]:
import peft
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,
    lora_alpha=1,  # facteur d'échelle, souvent fixé à 1
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",  # on ne fine-tune pas les biais
    task_type="CAUSAL_LM"
)

# On ajoute les couches adaptatrices (LoRA) au modèle de base
peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())


## Étape 5 : Configurer l'entraînement avec `TrainingArguments` et `Trainer`

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer
import os

output_directory = os.path.join("../cache", "peft_lab_outputs")

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,  # learning rate plus élevé que pour un fine-tuning complet
    num_train_epochs=1,
    use_cpu=True,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()


## Étape 6 : Sauvegarder le modèle LoRA optimisé

In [ ]:
import time

time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

print("Modèle LoRA sauvegardé dans :", peft_model_path)


## Étape 7 : Recharger le modèle LoRA pour l'inférence

On utilise `PeftModel.from_pretrained` avec `is_trainable=False` pour figer le modèle et faire uniquement de l'inférence.

In [ ]:
from peft import PeftModel

loaded_model = PeftModel.from_pretrained(
    foundation_model,
    peft_model_path,
    is_trainable=False,
)

loaded_model.eval()
print(loaded_model)


## Étape 8 : Générer du texte avec le modèle fine-tuné

In [ ]:
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")

outputs = loaded_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=30,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id,
)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))


### Comparaison avec le modèle de base (optionnel)

On peut comparer la génération du modèle de base (sans LoRA) à celle du modèle fine-tuné pour observer l'effet de l'adaptation.

In [ ]:
base_outputs = foundation_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=30,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id,
)

print("Base model:", tokenizer.batch_decode(base_outputs, skip_special_tokens=True))
print("LoRA model:", tokenizer.batch_decode(outputs, skip_special_tokens=True))
